# Extraction de Données de Contrats d'Assurance - Insurance Claims
## Pipeline de Préparation et Nettoyage des Données pour Deep Learning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Chargement des Données

In [ ]:
# Charger les données
df = pd.read_csv('Insurance claims data.csv')
print(f"Dimensions du dataset: {df.shape}")
print(f"\nPremières lignes:")
df.head()

## 2. Exploration et Contrôle Qualité

In [ ]:
# Informations générales
print("="*60)
print("INFORMATIONS GÉNÉRALES")
print("="*60)
df.info()

print("\n" + "="*60)
print("STATISTIQUES DESCRIPTIVES")
print("="*60)
df.describe().T

In [ ]:
# Analyse des valeurs manquantes
print("\n" + "="*60)
print("VALEURS MANQUANTES")
print("="*60)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Colonne': missing.index,
    'Manquantes': missing.values,
    'Pourcentage': missing_pct.values
}).sort_values('Pourcentage', ascending=False)
print(missing_df[missing_df['Manquantes'] > 0])

# Visualisation des valeurs manquantes
if missing_df['Manquantes'].sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=missing_df[missing_df['Manquantes'] > 0], 
                x='Colonne', y='Pourcentage', palette='viridis')
    plt.title('Pourcentage de Valeurs Manquantes par Colonne')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Détection des doublons et anomalies
print("\n" + "="*60)
print("DOUBLONS ET ANOMALIES")
print("="*60)
print(f"Nombre de doublons: {df.duplicated().sum()}")
print(f"Nombre de lignes uniques: {df.drop_duplicates().shape[0]}")

# Types de données
print("\n" + "="*60)
print("TYPES DE DONNÉES")
print("="*60)
print(df.dtypes)

## 3. Nettoyage des Données

In [ ]:
# Copie du dataset pour le nettoyage
df_clean = df.copy()

print("État initial:", df_clean.shape)

# 1. Supprimer les doublons
df_clean = df_clean.drop_duplicates()
print(f"Après suppression des doublons: {df_clean.shape}")

# 2. Traitement des valeurs manquantes
# Analyser les colonnes avec valeurs manquantes
missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()

if missing_cols:
    print(f"\nTraitement des colonnes avec valeurs manquantes: {missing_cols}")
    
    for col in missing_cols:
        if df_clean[col].dtype in ['float64', 'int64']:
            # Pour les colonnes numériques: imputer avec la médiane
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
            print(f"  {col}: remplissage avec la médiane")
        else:
            # Pour les colonnes catégorielles: imputer avec le mode
            df_clean[col].fillna(df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown', inplace=True)
            print(f"  {col}: remplissage avec le mode")

print(f"\nAprès traitement des valeurs manquantes: {df_clean.isnull().sum().sum()} valeurs manquantes restantes")

In [ ]:
# Détection et traitement des valeurs aberrantes (outliers)
print("\n" + "="*60)
print("DÉTECTION DES VALEURS ABERRANTES")
print("="*60)

numeric_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
    if len(outliers) > 0:
        print(f"\n{col}: {len(outliers)} valeurs aberrantes détectées")
        print(f"  Plage normale: [{lower_bound:.2f}, {upper_bound:.2f}]")
        # Garder les outliers mais les signaler (utile pour le deep learning)
        # On peut aussi les clamper si nécessaire
        # df_clean[col] = df_clean[col].clip(lower_bound, upper_bound)

## 4. Transformation et Préparation pour le Deep Learning

In [ ]:
# Identification des variables catégorielles et numériques
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()

print(f"Variables catégorielles: {len(categorical_cols)}")
print(f"  {categorical_cols}")
print(f"\nVariables numériques: {len(numeric_cols)}")
print(f"  {numeric_cols}")

# Encodage des variables catégorielles
print("\n" + "="*60)
print("ENCODAGE DES VARIABLES CATÉGORIELLES")
print("="*60)

df_transformed = df_clean.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_transformed[col] = le.fit_transform(df_transformed[col].astype(str))
    label_encoders[col] = le
    print(f"{col}: {len(le.classes_)} catégories")
    print(f"  Classes: {le.classes_[:10]}")  # Afficher les 10 premières

In [ ]:
# Normalisation des variables numériques
print("\n" + "="*60)
print("NORMALISATION DES VARIABLES NUMÉRIQUES")
print("="*60)

scaler = StandardScaler()
df_transformed[numeric_cols] = scaler.fit_transform(df_transformed[numeric_cols])

print(f"Normalisation complétée pour {len(numeric_cols)} colonnes")
print(f"\nAperçu après normalisation:")
print(df_transformed[numeric_cols].describe())

## 5. Division Train/Test

In [ ]:
# Division en ensembles d'entraînement et de test (80-20)
X_train, X_test = train_test_split(df_transformed, test_size=0.2, random_state=42)

print("="*60)
print("DIVISION TRAIN/TEST")
print("="*60)
print(f"Ensemble d'entraînement: {X_train.shape}")
print(f"Ensemble de test: {X_test.shape}")
print(f"Pourcentage train: {X_train.shape[0] / df_transformed.shape[0] * 100:.1f}%")
print(f"Pourcentage test: {X_test.shape[0] / df_transformed.shape[0] * 100:.1f}%")

## 6. Sauvegarde des Données Prétraitées

In [ ]:
import pickle
import os

# Créer un répertoire pour les données prétraitées
os.makedirs('data', exist_ok=True)

# Sauvegarder les ensembles d'entraînement et de test
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)

# Sauvegarder les scaler et encoders pour la prédiction future
with open('data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('data/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Sauvegarder aussi les informations des colonnes
with open('data/column_info.pkl', 'wb') as f:
    pickle.dump({
        'numeric_cols': numeric_cols,
        'categorical_cols': categorical_cols
    }, f)

print("="*60)
print("DONNÉES SAUVEGARDÉES")
print("="*60)
print("✓ data/X_train.csv")
print("✓ data/X_test.csv")
print("✓ data/scaler.pkl")
print("✓ data/label_encoders.pkl")
print("✓ data/column_info.pkl")

## 7. Résumé du Pipeline de Préparation

In [ ]:
print("="*60)
print("RÉSUMÉ DU PIPELINE DE PRÉPARATION DES DONNÉES")
print("="*60)

summary = f"""
✓ ÉTAPES COMPLÉTÉES:

1. Chargement: {df.shape[0]} lignes × {df.shape[1]} colonnes
   
2. Nettoyage:
   - Doublons supprimés: {df.shape[0] - df_clean.shape[0]}
   - Valeurs manquantes traitées
   - Valeurs aberrantes détectées

3. Transformation:
   - {len(categorical_cols)} variables catégorielles encodées
   - {len(numeric_cols)} variables numériques normalisées

4. Division des données:
   - Ensemble d'entraînement: {X_train.shape[0]} échantillons
   - Ensemble de test: {X_test.shape[0]} échantillons

5. Données finales:
   - Dimensions: {df_transformed.shape}
   - Type: Données numériques prêtes pour le Deep Learning
   - Scaler: StandardScaler
   - Encodeurs: LabelEncoder pour chaque variable catégorielles

📁 FICHIERS GÉNÉRÉS:
   - data/X_train.csv
   - data/X_test.csv
   - data/scaler.pkl
   - data/label_encoders.pkl
   - data/column_info.pkl

✓ PRÊT POUR LE DEEP LEARNING!
"""

print(summary)